# Laplace Composition Audit (NDIS + One-Run)

Clean simulation of a T-fold composition of (sub-sampled) Laplace mechanisms,
audited two ways:

1. **NDIS (Gaussian-vs-Gaussian via CLT)** — both with the *theoretical* per-step
   parameters and with parameters *estimated* from the simulated scores.
2. **One-run auditor** (`CanaryScoreAuditor.epsilon_one_run`).

Per-step model with sensitivity `mu`, scale `b`, sub-sampling rate `q`:

* H0 (out canary): `X_t ~ Laplace(0, b)`
* H1 (in  canary): `X_t ~ (1-q) Laplace(0, b) + q Laplace(mu, b)`

`q=1` reduces to a pure Laplace composition (e.g. tree mechanism, linear
contextual bandits).

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

project_dir = os.path.abspath(os.path.join(os.getcwd(), '../../'))
src_dir = os.path.join(project_dir, 'src')
sys.path.append(src_dir)

from auditing import CanaryScoreAuditor
from whitebox_auditing.ndis_1d import (
    ndis_eps_from_delta_1d_brentq,
    estimate_mean_variance,
)
from whitebox_auditing.laplace_composition import (
    sample_laplace,
    sample_laplace_mixture,
    eps_basic_composition,
    eps_advanced_composition,
    ndis_gaussian_params,
)

In [2]:
# ----- mechanism parameters -----
MU = 1.0          # per-step sensitivity / record contribution
B = 5.0           # Laplace scale (per step). Per-step pure-DP eps = MU/B.
Q = 0.1           # sub-sampling rate per step. Set Q = 1.0 for pure composition.
DELTA = 1e-5

# ----- audit parameters -----
N_OUT = 5000      # number of out canaries
N_IN  = 5000      # number of in  canaries
STEP_LIST = [10, 25, 50, 100, 200, 400, 800]
SEED = 0xC0FFEE

rng = np.random.default_rng(SEED)
print(f"per-step pure-DP eps   = mu/b      = {MU/B:.4f}")
print(f"per-step amplified eps = log(1+q(e^(mu/b)-1)) = "
      f"{np.log1p(Q*np.expm1(MU/B)):.4f}")

per-step pure-DP eps   = mu/b      = 0.2000
per-step amplified eps = log(1+q(e^(mu/b)-1)) = 0.0219


In [3]:
rows = []
for T in STEP_LIST:
    out_obs = sample_laplace(T, N_OUT, B, rng)
    in_obs, _ = sample_laplace_mixture(T, N_IN, Q, MU, B, rng)

    # 1D scalar scores: time-averaged sums (matches the convention used
    # in simulation_one_run_auditing.ipynb).
    out_scores = out_obs.sum(axis=1) / T
    in_scores  = in_obs.sum(axis=1)  / T

    # ---- theoretical upper bounds ----
    eps_basic = eps_basic_composition(T, b=B, mu=MU, q=Q)
    eps_adv   = eps_advanced_composition(T, b=B, mu=MU, q=Q, delta=DELTA)

    # ---- one-run lower bound (Steinke et al.) ----
    auditor = CanaryScoreAuditor(in_scores, out_scores)
    eps_one_run = auditor.epsilon_one_run(significance=0.05, delta=DELTA)

    # ---- NDIS with the *theoretical* CLT-Gaussian parameters ----
    th = ndis_gaussian_params(T, b=B, mu=MU, q=Q)
    if th['sigma1'] < th['sigma2']:
        eps_ndis_th = ndis_eps_from_delta_1d_brentq(
            sigma1=th['sigma1'], sigma2=th['sigma2'],
            mu1=th['mu1'], mu2=th['mu2'], delta_target=DELTA)
    else:
        eps_ndis_th = float('nan')  # degenerate (q == 1)

    # ---- NDIS with parameters *estimated* from the scores ----
    stats = estimate_mean_variance(in_scores * np.sqrt(T), out_scores * np.sqrt(T))
    sig1 = stats['out_std']
    sig2 = max(stats['in_std'], sig1 + 1e-5)  # NDIS requires sigma1 < sigma2
    eps_ndis_emp = ndis_eps_from_delta_1d_brentq(
        sigma1=sig1, sigma2=sig2,
        mu1=stats['out_mean'], mu2=stats['in_mean'],
        delta_target=DELTA)

    rows.append((T, eps_basic, eps_adv, eps_one_run, eps_ndis_th, eps_ndis_emp))
    print(f"T={T:>4d}  basic={eps_basic:6.3f}  adv={eps_adv:6.3f}  "
          f"one-run={eps_one_run:5.3f}  NDIS-th={eps_ndis_th:5.3f}  "
          f"NDIS-emp={eps_ndis_emp:5.3f}")

T=  10  basic= 0.219  adv= 0.219  one-run=0.000  NDIS-th=0.132  NDIS-emp=0.233
T=  25  basic= 0.547  adv= 0.538  one-run=0.000  NDIS-th=0.223  NDIS-emp=0.077
T=  50  basic= 1.095  adv= 0.767  one-run=0.011  NDIS-th=0.329  NDIS-emp=0.234
T= 100  basic= 2.190  adv= 1.099  one-run=0.076  NDIS-th=0.485  NDIS-emp=0.557
T= 200  basic= 4.380  adv= 1.583  one-run=0.237  NDIS-th=0.712  NDIS-emp=0.567
T= 400  basic= 8.759  adv= 2.296  one-run=0.255  NDIS-th=1.046  NDIS-emp=0.985
T= 800  basic=17.519  adv= 3.360  one-run=0.401  NDIS-th=1.539  NDIS-emp=1.539


In [ ]:
rows = np.array(rows)
Ts = rows[:, 0]
plt.figure(figsize=(9, 5.5))
#plt.plot(Ts, rows[:, 1], marker='o', label='upper: basic composition')
plt.plot(Ts, rows[:, 2], marker='o', label='upper: advanced composition')
plt.plot(Ts, rows[:, 3], marker='s', label='lower: one-run (Steinke)')
#plt.plot(Ts, rows[:, 4], marker='^', label='lower: NDIS (theoretical CLT)')
plt.plot(Ts, rows[:, 5], marker='v', label='lower: NDIS (empirical)')
plt.xscale('log')
plt.xlabel('T (number of composed Laplace steps)')
plt.ylabel('epsilon')
plt.title(f'Laplace composition  (mu={MU}, b={B}, q={Q}, delta={DELTA})')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()